# Cleaning and formatting cytokines data

In [13]:
import re
import polars as pl

In [14]:
# Load data
df = pl.read_csv(
    "../data/CV_plasma_Combined_cleaned_20260731.csv",
    infer_schema_length=None
)

In [15]:
# Variables mapping dict
cat_vars_mapping = {
    "Sex": {1: "male", 2: "female"},
    "Race": {
        1: "white_nonhispanic",
        2: "white_hispanic",
        3: "black_nonhispanic",
        4: "black_hispanic",
        5: "asian",
        6: "others",
    },
    "Tobaccouse": {1: "never", 2: "former", 3: "current"},
    "Alcoholuse": {1: "never", 2: "former", 3: "current"},
    "pathologicstage": {
        1: "stage_I",
        2: "stage_II",
        3: "stage_III",
        4: "stage_IVA",
        5: "stage_IVB"
    },
    "LVI": {0: "no", 1: "yes"},
    "PNI": {0: "no", 1: "yes"},
    "survival_status": {0: "alive", 1: "death"},
    "survival_2yrs": {0: "alive", 1: "death"}
}

In [16]:
# Remapping categorical variables to their corresponding values
df = df.with_columns(
    [pl.col(var).replace_strict(mapping).alias(var) for var, mapping in cat_vars_mapping.items()]
)

In [17]:
df["survival_2yrs"].value_counts()

survival_2yrs,count
str,u32
"""death""",61
"""alive""",151


In [18]:
df.head()

sample_names,sample_id,Cohort,EGF,IFNalpha2,IFNgamma,IL1alpha,IL1beta,IL4,IL6,IL8,IL10,MIP1alpha,TGFalpha,TNFalpha,TNFbeta,HBEGF,VEGFC,VEGFD,Ages,Sex,Race,Tobaccouse,Packyears,Alcoholuse,AverageUCSF,Spontaneouspain135,Functionalpain246,BPIPhysicalinterferences,BPIPsychologicalinterferences,BPI_intensity,Functionalscores,Symptomscore,GlobalhealthstatusQL2,QLQ30score,HN35,pathologicstage,LVI,PNI,survival_status,sx_to_lastfu,survival_2yrs,ss
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,str,str,f64,str,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,str,str,str,str,i64,str,i64
"""FF301C""","""FF301C""","""LLU""",29.45,25.24,90.0,6.4,3.95,4.21,3.54,4.14,3.92,25.93,3.32,18.74,10.46,83.27,381.47,263.05,67,"""female""","""white_nonhispanic""","""never""",0.0,"""never""",1.5,0.33,0.33,0.0,0,0.0,83.33,19.44,75.0,81.62,30.56,"""stage_I""","""no""","""no""","""alive""",1839,"""alive""",1
"""FF302C""","""FF302C""","""LLU""",71.5,9.93,46.35,0.0,0.0,2.33,3.42,4.53,75.2,2.9,1.52,25.54,3.74,75.2,923.75,237.53,68,"""male""","""white_nonhispanic""","""former""",52.5,"""former""",68.75,36.67,80.0,21.0,32,23.0,54.33,59.03,16.67,46.11,47.78,"""stage_IVA""","""no""","""yes""","""alive""",1721,"""alive""",4
"""FF303C""","""FF303C""","""LLU""",54.54,18.38,54.01,46.49,18.9,1.37,4.15,3.57,98.33,25.05,2.99,22.39,4.83,51.96,490.19,130.47,78,"""male""","""white_nonhispanic""","""never""",0.0,"""current""",67.5,63.33,63.33,9.0,4,22.0,39.67,31.25,50.0,57.56,45.68,"""stage_IVA""","""yes""","""no""","""death""",48,"""death""",4
"""FF304C""","""FF304C""","""LLU""",132.67,27.47,81.82,6.89,3.95,4.56,9.69,7.29,20.63,20.32,4.31,28.18,9.32,85.47,1148.22,34.22,49,"""male""","""white_nonhispanic""","""current""",72.0,"""current""",95.0,86.67,100.0,0.0,0,29.0,0.0,68.75,0.0,19.23,94.44,"""stage_IVA""","""no""","""yes""","""death""",131,"""death""",3
"""FF305C""","""FF305C""","""LLU""",28.01,13.63,66.2,5.44,0.92,3.04,2.42,5.44,10.63,17.14,3.98,39.61,5.94,71.54,1113.45,142.55,65,"""female""","""white_nonhispanic""","""former""",42.0,"""never""",37.0,11.0,33.67,2.0,10,18.0,82.67,27.08,75.0,76.67,17.44,"""stage_II""","""no""","""no""","""alive""",1679,"""alive""",1


In [19]:
# df.schema
df.null_count()

sample_names,sample_id,Cohort,EGF,IFNalpha2,IFNgamma,IL1alpha,IL1beta,IL4,IL6,IL8,IL10,MIP1alpha,TGFalpha,TNFalpha,TNFbeta,HBEGF,VEGFC,VEGFD,Ages,Sex,Race,Tobaccouse,Packyears,Alcoholuse,AverageUCSF,Spontaneouspain135,Functionalpain246,BPIPhysicalinterferences,BPIPsychologicalinterferences,BPI_intensity,Functionalscores,Symptomscore,GlobalhealthstatusQL2,QLQ30score,HN35,pathologicstage,LVI,PNI,survival_status,sx_to_lastfu,survival_2yrs,ss
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,100,0,0,100,0,0,0,0,0,0,77,0,70,70,70,71,71,70,70,70,70,70,71,0,0,0,0,0,0,0


In [20]:
df.glimpse()

Rows: 212
Columns: 43
$ sample_names                  <str> 'FF301C', 'FF302C', 'FF303C', 'FF304C', 'FF305C', 'FF306C', 'FF307C', 'FF308C', 'FF310C', 'FF312C'
$ sample_id                     <str> 'FF301C', 'FF302C', 'FF303C', 'FF304C', 'FF305C', 'FF306C', 'FF307C', 'FF308C', 'FF310C', 'FF312C'
$ Cohort                        <str> 'LLU', 'LLU', 'LLU', 'LLU', 'LLU', 'LLU', 'LLU', 'LLU', 'LLU', 'LLU'
$ EGF                           <f64> 29.45, 71.5, 54.54, 132.67, 28.01, 66.25, 76.93, 16.56, 36.06, 65.71
$ IFNalpha2                     <f64> 25.24, 9.93, 18.38, 27.47, 13.63, 19.54, 7.36, 0.04, 18.38, 0.04
$ IFNgamma                      <f64> 90.0, 46.35, 54.01, 81.82, 66.2, 54.01, 21.11, 37.59, 31.81, 34.84
$ IL1alpha                      <f64> 6.4, 0.0, 46.49, 6.89, 5.44, 2.12, 1.21, 0.0, 12.32, 0.0
$ IL1beta                       <f64> 3.95, 0.0, 18.9, 3.95, 0.92, 1.38, 0.0, 0.0, 4.57, 0.0
$ IL4                           <f64> 4.21, 2.33, 1.37, 4.56, 3.04, 1.97, 1.01, 0.57, 1.37, 0.

In [21]:
df.describe()

statistic,sample_names,sample_id,Cohort,EGF,IFNalpha2,IFNgamma,IL1alpha,IL1beta,IL4,IL6,IL8,IL10,MIP1alpha,TGFalpha,TNFalpha,TNFbeta,HBEGF,VEGFC,VEGFD,Ages,Sex,Race,Tobaccouse,Packyears,Alcoholuse,AverageUCSF,Spontaneouspain135,Functionalpain246,BPIPhysicalinterferences,BPIPsychologicalinterferences,BPI_intensity,Functionalscores,Symptomscore,GlobalhealthstatusQL2,QLQ30score,HN35,pathologicstage,LVI,PNI,survival_status,sx_to_lastfu,survival_2yrs,ss
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,str,f64
"""count""","""212""","""212""","""212""",212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,112.0,212.0,212.0,112.0,212.0,212.0,212.0,"""212""","""212""","""212""",135.0,"""212""",142.0,142.0,142.0,141.0,141.0,142.0,142.0,142.0,142.0,142.0,141.0,"""212""","""212""","""212""","""212""",212.0,"""212""",212.0
"""null_count""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,100.0,0.0,0.0,0.0,"""0""","""0""","""0""",77.0,"""0""",70.0,70.0,70.0,71.0,71.0,70.0,70.0,70.0,70.0,70.0,71.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,36.451226,50.149434,52.32967,31.910047,4.660519,185.573255,13.037358,12.72467,39.947783,14.475849,11.033482,16.895755,35.916887,51.617232,412.086321,176.559481,62.886792,null,null,null,15.965185,null,44.022606,36.070845,47.615282,8.358156,16.496454,19.873239,69.901549,29.539014,54.518944,70.252394,34.163191,null,null,null,null,908.301887,null,2.79717
"""std""",null,null,null,39.267355,47.94639,39.002566,67.828514,9.58651,604.437024,24.826131,24.838677,78.075083,46.602309,39.702632,9.270465,101.20653,27.184616,308.239584,301.379999,14.000556,null,null,null,23.72698,null,30.707376,30.593177,33.527031,8.57828,13.919475,12.005973,26.519332,22.19019,28.226752,23.030657,20.126162,null,null,null,null,554.44003,null,1.366974
"""min""","""10""","""FF301C""","""LLU""",0.0,0.0,6.58,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,4.52,0.33,0.0,0.0,0.0,22.0,"""female""","""asian""","""current""",0.0,"""current""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.87,0.0,"""stage_I""","""no""","""no""","""alive""",15.0,"""alive""",0.0
"""25%""",null,null,null,13.66,8.66,30.12,1.66,0.0,1.37,2.29,3.41,9.39,5.43,0.73,11.62,3.58,35.5,175.52,57.7,54.0,null,null,null,0.0,null,15.0,6.67,16.67,0.0,1.0,11.0,54.0,10.42,33.33,56.67,17.75,null,null,null,null,414.0,null,2.0
"""50%""",null,null,null,21.88,34.05,45.44,9.64,2.36,6.58,4.48,5.31,15.59,9.24,2.07,14.65,7.33,44.42,329.69,100.54,65.0,null,null,null,0.0,null,44.5,29.67,50.0,6.0,15.0,22.0,77.0,27.08,58.33,76.07,33.21,null,null,null,null,846.0,null,3.0
"""75%""",null,null,null,48.81,83.26,67.08,18.3,4.36,55.99,11.83,8.89,38.41,13.55,3.98,19.13,13.09,62.81,545.87,187.12,72.0,null,null,null,30.0,null,68.75,60.0,76.67,15.0,28.0,28.0,93.33,43.75,83.33,89.49,51.85,null,null,null,null,1470.0,null,4.0
"""max""","""P-039""","""S0080""","""UAB""",283.4,288.54,454.35,552.31,70.37,5864.0,199.75,208.34,706.6,663.14,277.39,66.6,855.53,155.91,1516.24,3497.0,97.0,"""male""","""white_nonhispanic""","""never""",120.0,"""never""",100.0,100.0,100.0,30.0,40.0,40.0,100.0,95.83,100.0,100.0,94.44,"""stage_IVB""","""yes""","""yes""","""death""",1917.0,"""death""",6.0


In [22]:
# Clean columnn names
df = df.rename(
    {
        col: re.sub(r"\s+", "_", re.sub(r"[^\w\s]", "", col.strip().lower()))
        for col in df.columns
    }
)

In [23]:
# Recheck the data after cleaning
df.describe()

statistic,sample_names,sample_id,cohort,egf,ifnalpha2,ifngamma,il1alpha,il1beta,il4,il6,il8,il10,mip1alpha,tgfalpha,tnfalpha,tnfbeta,hbegf,vegfc,vegfd,ages,sex,race,tobaccouse,packyears,alcoholuse,averageucsf,spontaneouspain135,functionalpain246,bpiphysicalinterferences,bpipsychologicalinterferences,bpi_intensity,functionalscores,symptomscore,globalhealthstatusql2,qlq30score,hn35,pathologicstage,lvi,pni,survival_status,sx_to_lastfu,survival_2yrs,ss
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,str,f64
"""count""","""212""","""212""","""212""",212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,112.0,212.0,212.0,112.0,212.0,212.0,212.0,"""212""","""212""","""212""",135.0,"""212""",142.0,142.0,142.0,141.0,141.0,142.0,142.0,142.0,142.0,142.0,141.0,"""212""","""212""","""212""","""212""",212.0,"""212""",212.0
"""null_count""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,100.0,0.0,0.0,0.0,"""0""","""0""","""0""",77.0,"""0""",70.0,70.0,70.0,71.0,71.0,70.0,70.0,70.0,70.0,70.0,71.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,36.451226,50.149434,52.32967,31.910047,4.660519,185.573255,13.037358,12.72467,39.947783,14.475849,11.033482,16.895755,35.916887,51.617232,412.086321,176.559481,62.886792,null,null,null,15.965185,null,44.022606,36.070845,47.615282,8.358156,16.496454,19.873239,69.901549,29.539014,54.518944,70.252394,34.163191,null,null,null,null,908.301887,null,2.79717
"""std""",null,null,null,39.267355,47.94639,39.002566,67.828514,9.58651,604.437024,24.826131,24.838677,78.075083,46.602309,39.702632,9.270465,101.20653,27.184616,308.239584,301.379999,14.000556,null,null,null,23.72698,null,30.707376,30.593177,33.527031,8.57828,13.919475,12.005973,26.519332,22.19019,28.226752,23.030657,20.126162,null,null,null,null,554.44003,null,1.366974
"""min""","""10""","""FF301C""","""LLU""",0.0,0.0,6.58,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,4.52,0.33,0.0,0.0,0.0,22.0,"""female""","""asian""","""current""",0.0,"""current""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.87,0.0,"""stage_I""","""no""","""no""","""alive""",15.0,"""alive""",0.0
"""25%""",null,null,null,13.66,8.66,30.12,1.66,0.0,1.37,2.29,3.41,9.39,5.43,0.73,11.62,3.58,35.5,175.52,57.7,54.0,null,null,null,0.0,null,15.0,6.67,16.67,0.0,1.0,11.0,54.0,10.42,33.33,56.67,17.75,null,null,null,null,414.0,null,2.0
"""50%""",null,null,null,21.88,34.05,45.44,9.64,2.36,6.58,4.48,5.31,15.59,9.24,2.07,14.65,7.33,44.42,329.69,100.54,65.0,null,null,null,0.0,null,44.5,29.67,50.0,6.0,15.0,22.0,77.0,27.08,58.33,76.07,33.21,null,null,null,null,846.0,null,3.0
"""75%""",null,null,null,48.81,83.26,67.08,18.3,4.36,55.99,11.83,8.89,38.41,13.55,3.98,19.13,13.09,62.81,545.87,187.12,72.0,null,null,null,30.0,null,68.75,60.0,76.67,15.0,28.0,28.0,93.33,43.75,83.33,89.49,51.85,null,null,null,null,1470.0,null,4.0
"""max""","""P-039""","""S0080""","""UAB""",283.4,288.54,454.35,552.31,70.37,5864.0,199.75,208.34,706.6,663.14,277.39,66.6,855.53,155.91,1516.24,3497.0,97.0,"""male""","""white_nonhispanic""","""never""",120.0,"""never""",100.0,100.0,100.0,30.0,40.0,40.0,100.0,95.83,100.0,100.0,94.44,"""stage_IVB""","""yes""","""yes""","""death""",1917.0,"""death""",6.0


In [24]:
# Writing cleaned and formatted cytokines data
df.write_csv(
    "../data/CV_plasma_Combined_formatted_20260731.csv",
    separator=",",
    include_header=True,
    null_value="",
    quote_style="necessary"
)